# VIS model 

In [6]:

from utils import get_device
from data_process_v4 import get_loaders, class_cols
from data_process_v4 import TASK_3_TEST_LABELS_DIR

from VIS_V1 import create_model, train_model
from utils import get_device
import torch
import pandas as pd
from tqdm import tqdm

In [7]:
device = get_device()

loaders = get_loaders(
    image_size=(384, 384),
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)


train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders
class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7


class_counts = [int(train_df[c].sum()) for c in class_cols_order]
model_name = "vis_model.pth"
base_model = create_model(num_classes=num_classes, model_name="vit_base_patch16_384")
ckpt = torch.load(model_name, map_location=device)
base_model.load_state_dict(ckpt['model_state'])

<All keys matched successfully>

In [5]:

model, history2 = train_model(
    train_loader,
    val_loader,
    num_classes=num_classes,
    device=device,
    epochs=10,                 
    class_counts=class_counts,
    model=model,          
    save_path=model_name
)


Epoch 1/10 train_loss=0.0866 val_loss=2.1891 train_acc=0.2203 val_acc=0.2176
  ✓ New best val_acc=0.2176, model saved to vis_model.pth


Epoch 2/10 train_loss=0.0887 val_loss=1.6548 train_acc=0.2330 val_acc=0.3886
  ✓ New best val_acc=0.3886, model saved to vis_model.pth


Epoch 3/10 train_loss=0.0965 val_loss=2.8275 train_acc=0.2229 val_acc=0.1503


KeyboardInterrupt: 

In [4]:
import torch
from eval_vis import evaluate_model

# device (can also reuse what you used in train_model)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
base_model.to(device)


metrics, cm, (y_true, y_pred) = evaluate_model(
    model=base_model,
    data_loader=test_loader,
    device=device,
    class_names=class_cols_order,   # or None
    num_classes=len(class_cols_order),  # or None to infer
)

print("Metrics dict:", metrics)


Evaluating:   0%|          | 0/95 [00:00<?, ?it/s]

Loss: 1.9640 | Accuracy: 0.3664 | Mean (macro) recall: 0.4954


Metrics dict: {'loss': 1.9639554761704945, 'accuracy': 0.3664021164021164, 'mean_recall': 0.4954290379655319}


# resnet

In [7]:
import torch
from resnet_v2 import train_model, create_resnet_model
from eval_resnet import evaluate_model
from data_process_v4 import get_loaders, class_cols
from data_process_v4 import TASK_3_TEST_LABELS_DIR
from utils import get_device

In [ ]:
device = get_device()
loaders = get_loaders(
    image_size=(650, 400),                 
    num_workers=0,
    test_csv_path=TASK_3_TEST_LABELS_DIR,
)

train_loader, val_loader, test_loader, train_df, val_df, test_df = loaders

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
num_classes = 7

# class distribution for BalancedFocalLoss
class_counts = [int(train_df[c].sum()) for c in class_cols_order]

model = create_resnet_model(
     num_classes=num_classes,
     pretrained=True,
     backbone_name="resnet50",
     head_hidden_dim=512,
     head_dropout=0.3,
 )

In [ ]:


model, history = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=num_classes,
    epochs=40,
    lr=3e-4,
    weight_decay=0.05,
    class_counts=class_counts,              
    save_path="best_resnet50_resizer.pth",
    device=device,                          
    backbone_name="resnet50",
    head_hidden_dim=512,
    head_dropout=0.3,
)


In [12]:
device = get_device()
base_model  = create_resnet_model(
     num_classes=7,
     pretrained=True,
     backbone_name="resnet50",
     head_hidden_dim=512,
     head_dropout=0.3,
 )
ckpt = torch.load("best_resnet50_resizer.pth", map_location=device)
base_model.load_state_dict(ckpt["model_state"])

<All keys matched successfully>

In [13]:
from eval_resnet import evaluate_model

device = get_device()
print("Using device:", device)

class_cols_order = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]

# evaluate on test set, compute mean recall and plot confusion matrix
mean_recall, cm, (y_true, y_pred) = evaluate_model(
    model=base_model,
    data_loader=test_loader,
    device=device,
    class_names=class_cols_order,   
    num_classes=len(class_cols_order),
    verbose=True,
)

print("Mean recall (macro):", mean_recall)

Using device: mps


Mean (macro) recall: 0.6755


Mean recall (macro): 0.6754721858151264
